In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

cwd = Path.cwd().resolve()
candidate_paths = [cwd, cwd.parent, cwd.parent.parent]

for candidate in candidate_paths:
    data_dir = candidate / "NeurIPS_2026" / "data"
    if data_dir.exists():
        sys.path.insert(0, str(data_dir))
        break
else:
    if (cwd / "synthetic_cyclic_scm.py").exists():
        sys.path.insert(0, str(cwd))

from synthetic_cyclic_scm import (
    generate_synthetic_scm_linear,

)
from synthetic_cyclic_scm_nonlinear import (
    generate_synthetic_scm_nonlinear,
)

In [ ]:
X_NODE = 0
Y_NODE = 1
CYCLIC_FLAG = True


def adjacency_to_digraph(W: np.ndarray) -> nx.DiGraph:
    """Convert an SCM adjacency matrix into a NetworkX DiGraph."""
    graph = nx.DiGraph()
    n_nodes = W.shape[0]
    graph.add_nodes_from(range(n_nodes))

    for child in range(n_nodes):
        for parent in np.flatnonzero(W[child]):
            weight = float(W[child, parent])
            graph.add_edge(int(parent), int(child), weight=weight, abs_weight=abs(weight))

    return graph


def draw_scm(ax, W: np.ndarray, title: str, x_node: int = X_NODE, y_node: int = Y_NODE, draw_with_edge_weights: str = None) -> None:
    graph = adjacency_to_digraph(W)
    positions = nx.kamada_kawai_layout(graph, weight=draw_with_edge_weights)
    labels = {
        node: ("X" if node == x_node else "Y" if node == y_node else f"Z{node}")
        for node in graph.nodes
    }
    edge_labels = {(u, v): f"{data['weight']:.2f}" for u, v, data in graph.edges(data=True)}
    node_colors = [
        "#c44e52" if node == x_node else "#4c72b0" if node == y_node else "#e9eef5"
        for node in graph.nodes
    ]
    edge_widths = [1.5 + 2.5 * graph[u][v]["abs_weight"] for u, v in graph.edges]
    edge_colors = ["#2f3e46" if graph[u][v]["weight"] > 0 else "#8d99ae" for u, v in graph.edges]

    nx.draw_networkx_nodes(
        graph,
        pos=positions,
        ax=ax,
        node_color=node_colors,
        node_size=1450,
        edgecolors="#1f2933",
        linewidths=1.4,
    )
    nx.draw_networkx_labels(
        graph,
        pos=positions,
        ax=ax,
        labels=labels,
        font_size=12,
        font_weight="bold",
    )
    nx.draw_networkx_edges(
        graph,
        pos=positions,
        ax=ax,
        edge_color=edge_colors,
        width=edge_widths,
        arrows=True,
        arrowsize=18,
        arrowstyle="-|>",
        connectionstyle="arc3,rad=0.08",
        min_source_margin=18,
        min_target_margin=18,
        alpha=0.9,
    )
    nx.draw_networkx_edge_labels(
        graph,
        pos=positions,
        edge_labels=edge_labels,
        ax=ax,
        font_size=8,
        rotate=False,
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.85, "pad": 0.2},
    )

    ax.set_title(title, fontsize=13, fontweight="bold", pad=10)
    ax.set_axis_off()
    ax.set_facecolor("#fbfbfd")


linear_draws = [
    generate_synthetic_scm_linear(
        n_nodes=8,
        n_edges=12,
        n_samples=200,
        n_observed=5,
        cyclic_flag=True,
        pretreatment_flag=True,
        x_singleton_scc_flag=False,
        allow_two_cycles=False,
        sigma_range=[0.3, 0.7],
        treatment=X_NODE,
        outcome=Y_NODE,
        treatment_outcome_edge=True,
        require_weak_connectivity=True,
        sigma_backdoor_flag=False,
        max_graph_attempts=1000,
        rng_seed=seed,
    )
    for seed in (29,20)
]


nonlinear_draws = [
    generate_synthetic_scm_nonlinear(
        n_nodes=12,
        n_edges=20,
        n_samples=200,
        n_observed=4,
        target_operator_norm=0.8,
        sigma_range=[0.3, 0.7],
        rng_seed=seed,
        cyclic_flag=False,
        allow_two_cycles=False,
        pretreatment_flag=True,
        treatment=X_NODE,
        outcome=Y_NODE,
        x_singleton_scc_flag=True,
        treatment_outcome_edge=True,
        require_weak_connectivity=True,
        sigma_backdoor_flag=False,
        max_graph_attempts=1000,
    )
    for seed in (5, 212)
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12), facecolor="#f4f6f8")

for idx, dataset in enumerate(linear_draws):
    draw_scm(axes[0, idx], dataset.W, f"Linear draw {idx + 1}", draw_with_edge_weights="abs_weight")

for idx, dataset in enumerate(nonlinear_draws):
    draw_scm(axes[1, idx], dataset.W, f"Nonlinear draw {idx + 1}")

graph_family = "Cyclic" if CYCLIC_FLAG else "Acyclic"
fig.suptitle(f"Synthetic {graph_family} SCM draws", fontsize=18, fontweight="bold", y=0.98)
fig.tight_layout(pad=2.0)
plt.show()

NameError: name 'np' is not defined